# Phase 0 Colab bring-up

Run on a CPU Colab runtime before Phase 1. This verifies pinned packages, Drive paths, frozen manifest hashes, front-end parity, raw paths, and train/validation loaders. The test manifest is hashed as bytes; test paths are checked only for existence. No test features, labels, caches, audio, or scores are loaded.

Code commit for this preflight: 075c1b023bd2314b7316ed64a932e1e1317bd805.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, json, hashlib, platform

from google.colab import drive
drive.mount("/content/drive")

CODE_COMMIT = "075c1b023bd2314b7316ed64a932e1e1317bd805"
REPOSITORY_URL = "https://github.com/NasrallahRAFI/Functional-Project.git"
CODE_ROOT = Path("/content/Functional-Project")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/drone-ml-project")
if not DRIVE_PROJECT_ROOT.is_dir():
    raise RuntimeError(f"STOP C0.5: Drive project root is missing: {DRIVE_PROJECT_ROOT}")
STATE={"code_commit":CODE_COMMIT,"started_utc":datetime.now(timezone.utc).isoformat(),
       "gates":[],"test_reads":0}
REPORT_TABLES=DRIVE_PROJECT_ROOT/"results"/"tables"
def write_report():
    STATE["finished_utc"]=datetime.now(timezone.utc).isoformat()
    STATE["test_reads"]=0
    REPORT_TABLES.mkdir(parents=True,exist_ok=True)
    json_path=REPORT_TABLES/"run_report_phase0.json"
    md_path=REPORT_TABLES/"run_report_phase0.md"
    json_path.write_text(json.dumps(STATE,indent=2)+"\n",encoding="utf-8")
    md_path.write_text("# Phase 0 Colab bring-up\n\n"+json.dumps(STATE,indent=2)+"\n",encoding="utf-8")
    print("Phase 0 report written:",json_path,md_path)
def record(gate,status,evidence):
    item={"gate":gate,"status":status,"evidence":evidence}
    STATE["gates"].append(item)
    print(gate,status,evidence)
def stop(gate,evidence):
    record(gate,"STOP",evidence)
    STATE["stop"]={"gate":gate,"evidence":evidence}
    write_report()
    raise RuntimeError(f"STOP {gate}: {evidence}")
def require(gate,ok,evidence):
    record(gate,"PASS" if ok else "STOP",evidence)
    if not ok:
        STATE["stop"]={"gate":gate,"evidence":evidence}
        write_report()
        raise RuntimeError(f"STOP {gate}: {evidence}")
try:
    if CODE_ROOT.exists():
        subprocess.run(["git","-C",str(CODE_ROOT),"fetch","--tags","origin"],check=True)
    else:
        subprocess.run(["git","clone",REPOSITORY_URL,str(CODE_ROOT)],check=True)
    subprocess.run(["git","-C",str(CODE_ROOT),"checkout","--detach",CODE_COMMIT],check=True)
    actual_commit=subprocess.check_output(["git","-C",str(CODE_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual_commit != CODE_COMMIT:
        stop("C0.5 code commit", {"actual":actual_commit,"expected":CODE_COMMIT})
    TAG_PRESENT=bool(subprocess.check_output(["git","-C",str(CODE_ROOT),"tag","--list","frontend-core-frozen"],text=True).strip())
    subprocess.run([sys.executable,"-m","pip","install","-r",str(CODE_ROOT/"requirements.txt")],check=True)
except Exception as exc:
    if "stop" in STATE:
        raise
    stop("C0.5 checkout or pinned install", {"error_type":type(exc).__name__,"error":str(exc)})
sys.path.insert(0,str(CODE_ROOT/"src"))
os.chdir(CODE_ROOT)


In [ ]:
import importlib.metadata as md
import numpy as np, pandas as pd, scipy, soundfile as sf, soxr, librosa, sklearn, torch, yaml

pins={}
for line in (CODE_ROOT/"requirements.txt").read_text(encoding="utf-8").splitlines():
    line=line.strip()
    if line and not line.startswith("#") and "==" in line:
        name,version=line.split("==",1)
        pins[name.lower().replace("_","-")]=version
package_versions={name:md.version(name) for name in pins}
loaded_modules={"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__,
                "soundfile":sf.__version__,"soxr":soxr.__version__,"librosa":librosa.__version__,
                "scikit-learn":sklearn.__version__,"pyyaml":yaml.__version__}
versions={**package_versions,**loaded_modules,"torch":torch.__version__}
mismatch={name:{"installed":package_versions[name],"pinned":pin}
          for name,pin in pins.items() if package_versions[name]!=pin}
module_mismatch={name:{"loaded":value,"installed":package_versions[name]}
                 for name,value in loaded_modules.items() if value!=package_versions[name]}
if mismatch or module_mismatch:
    print("RESTART RUNTIME",{"pin_mismatch":mismatch,"loaded_module_mismatch":module_mismatch})
    stop("C0.5 package versions", {"pin_mismatch":mismatch,"loaded_module_mismatch":module_mismatch})
if torch.cuda.is_available():
    stop("C0.5 CPU device", "CUDA is available; reconnect to a CPU runtime.")
cpu_count=os.cpu_count()
if not cpu_count: stop("C0.5 CPU count", "CPU count unavailable.")
torch.set_num_threads(cpu_count)
try:
    import psutil
    ram_bytes=int(psutil.virtual_memory().total)
except ImportError:
    with open("/proc/meminfo",encoding="utf-8") as f:
        ram_bytes=int(next(x.split()[1] for x in f if x.startswith("MemTotal:")))*1024
ENV={"python":sys.version,"platform":platform.platform(),"libraries":versions,
     "cpu_count":cpu_count,"ram_bytes":ram_bytes,"device":"cpu",
     "torch_threads":torch.get_num_threads(),"code_commit":actual_commit}
from dronedet.config import load_project_config, params_hash, cache_content_hash, FrontEndConfig
from dataclasses import replace
from dronedet.bundle import verify_bundle
PROJECT=load_project_config(CODE_ROOT/"config.yaml")
P=PROJECT["P"]
if Path(P["root"]).resolve()!=DRIVE_PROJECT_ROOT.resolve():
    stop("C0.5 Drive profile", f"unexpected Colab project root {P['root']}")
interim=P["interim"]
old=json.loads((interim/"environment_log.json").read_text(encoding="utf-8"))
ENV["differences_from_data_preparation"]={
    k:{"data_preparation":old.get("packages",{}).get(k),"colab":v}
    for k,v in versions.items() if old.get("packages",{}).get(k)!=v}
target=interim/"environment_colab.json"
tmp=target.with_name(target.name+".tmp")
tmp.write_text(json.dumps(ENV,indent=2)+"\n",encoding="utf-8")
os.replace(tmp,target)
record("C0.5 runtime/pins/Drive profile","PASS",
       {"device":"cpu","cpu_count":cpu_count,"threads":torch.get_num_threads(),"environment":str(target)})

In [ ]:
# Manifest integrity: SHA-256 over bytes only; do not parse the test manifest here.
hash_paths={"manifest_full":P["interim"]/"manifest_full.csv",
            "train":P["splits"]/"train.csv","val":P["splits"]/"val.csv","test":P["splits"]/"test.csv"}
required_integrity_files=[*hash_paths.values(),P["splits"]/"FROZEN.txt",
    P["tables"]/"manifest_hashes.json",P["spec"]/"frontend_hash.json",
    P["spec"]/"cache_content_hash.json",P["spec"]/"norm_stats_global.npz",
    P["golden"]/"report_filterbank.npy",P["golden"]/"frontend_bundle"/"hash.txt"]
missing_integrity=[str(path) for path in required_integrity_files if not path.is_file()]
if missing_integrity:
    stop("C0.6 required integrity files", {"missing":missing_integrity})
recorded=json.loads((P["tables"]/"manifest_hashes.json").read_text(encoding="utf-8"))
yaml_hashes=PROJECT["split"]["manifest_hashes"]
frozen={}
for line in (P["splits"]/"FROZEN.txt").read_text(encoding="utf-8").splitlines():
    if ": " in line:
        k,v=line.split(": ",1); frozen[k.strip()]=v.strip()
hashes={}
for name,path in hash_paths.items():
    actual=hashlib.sha256(path.read_bytes()).hexdigest()
    if name=="manifest_full": frozen_key="manifest_full"
    else: frozen_key=name
    expected=[recorded.get(name),yaml_hashes.get(name),frozen.get(frozen_key)]
    if not all(x==actual for x in expected):
        stop("C0.6 manifest hashes", {"name":name,"actual":actual,"expected":expected})
    hashes[name]=actual
record("C0.6 manifest hashes","PASS",hashes)

fe=PROJECT["fe"]
front=json.loads((P["spec"]/"frontend_hash.json").read_text(encoding="utf-8"))
param_hash=params_hash(fe)
content_hash=cache_content_hash(fe)
content_record=json.loads((P["spec"]/"cache_content_hash.json").read_text(encoding="utf-8"))
if content_record.get("cache_content_hash")!=content_hash or content_record.get("params_hash")!=params_hash(fe):
    stop("C0.3 recorded cache content hash", {"actual_content_hash":content_hash,
         "recorded_content_hash":content_record.get("cache_content_hash"),
         "actual_params_hash":params_hash(fe),"recorded_params_hash":content_record.get("params_hash")})
record("C0.3 recorded cache content hash","PASS",content_record)
mode_invariant=(cache_content_hash(replace(fe,mode="pcen"))==content_hash)
pcen_invariant=all(cache_content_hash(replace(fe,**{k:v}))==content_hash for k,v in {
    "pcen_time_constant":0.5,"pcen_gain":0.9,"pcen_bias":3.0,
    "pcen_power":0.7,"pcen_eps":1e-5,"pcen_input_scale":2**30}.items())
if not mode_invariant or not pcen_invariant:
    stop("C0.3 cache content hash", {"mode_invariant":mode_invariant,"pcen_invariant":pcen_invariant})
record("C0.3 cache content hash","PASS",{"cache_content_hash":content_hash,
       "mode_excluded":mode_invariant,"pcen_keys_excluded":pcen_invariant})
if param_hash!=front.get("params_hash") or param_hash!=old.get("frontend_params_hash"):
    stop("C0.6 params hash", {"actual":param_hash,"frontend_hash":front.get("params_hash"),"environment_hash":old.get("frontend_params_hash")})
stats=P["spec"]/"norm_stats_global.npz"
stats_expected=front.get("artifacts",{}).get("data/processed/spectrograms/norm_stats_global.npz",{}).get("sha256")
stats_actual=hashlib.sha256(stats.read_bytes()).hexdigest()
if not stats_expected or stats_actual!=stats_expected:
    stop("C0.6 normalization statistics hash", {"actual":stats_actual,"expected":stats_expected})
bundle=verify_bundle(P["golden"]/"frontend_bundle",fe)
reference=FrontEndConfig.reference()
saved=np.load(P["golden"]/"report_filterbank.npy",allow_pickle=False)
bank=librosa.filters.mel(sr=reference.sr,n_fft=reference.n_fft,n_mels=reference.n_mels,
    fmin=reference.fmin,fmax=reference.fmax,htk=True,norm="slaney",dtype=np.float32)
support=np.array_equal(saved>0,bank>0)
nz=saved!=0
rel=float(np.max(np.abs(saved[nz]-bank[nz])/np.maximum(np.abs(saved[nz]),1e-30)))
if not support or rel>1e-6:
    stop("C0.6 filterbank parity", {"support_equal":support,"relative_max":rel})
record("C0.6 front-end parity","PASS",
       {"params_hash":param_hash,"stats_sha256":stats_actual,"bundle":bundle,
        "filterbank_support_equal":support,"filterbank_relative_max":rel})
record("C0.3 frontend-core-frozen tag","PASS" if TAG_PRESENT else "BLOCKED",
       {"tag_present":TAG_PRESENT,"notebook_30_blocked_until_tag":not TAG_PRESENT})

In [ ]:
import pandas as pd
from dronedet.splits import load_split
from dronedet.features import load_tabular

raw=Path(P["raw_root"])
roots=[raw/"Binary_Drone_Audio",raw/"DroneAudioDataset-master"/"Binary_Drone_Audio"]
binary=next((x for x in roots if x.is_dir()),None)
if binary is None: stop("C0.7 raw paths", f"Binary_Drone_Audio missing under {raw}")
path_counts={}
for split in ("train","val","test"):
    # For test, this cell checks path existence only; it does not inspect audio or labels.
    m=pd.read_csv(P["splits"]/f"{split}.csv",keep_default_na=False)
    missing=[]; escaped=[]
    for cid in m["clip_id"]:
        resolved=(binary/Path(*str(cid).replace("\\","/").split("/"))).resolve()
        if binary.resolve() not in resolved.parents: escaped.append(str(cid))
        if not resolved.is_file(): missing.append(str(cid))
    path_counts[split]={"manifest_rows":len(m),"missing":len(missing),"outside_root":len(escaped)}
    if missing or escaped: stop("C0.7 raw paths", {"split":split,"missing_sample":missing[:5],"outside_root_sample":escaped[:5]})
record("C0.7 raw paths","PASS",path_counts)

smoke={}
for split,n in (("train",7607),("val",1510)):
    m=load_split(P,split); a=load_tabular(P,split)
    index=pd.read_csv(P["spec"]/f"index_{split}.csv",keep_default_na=False)
    if len(m)!=n or a["X"].shape!=(n,84) or not np.isfinite(a["X"]).all():
        stop("C0.8 loader smoke", {"split":split,"shape":list(a["X"].shape),"finite":bool(np.isfinite(a["X"]).all())})
    if not np.array_equal(m.clip_id.to_numpy(),a["clip_id"]) or not np.array_equal(m.clip_id.to_numpy(),index.clip_id.to_numpy()):
        stop("C0.8 loader smoke", {"split":split,"row_order_match":False})
    flags={}
    for flag in ("near_silent","partial_zero","clipped_flag","dc_flag"):
        if flag not in a or not np.array_equal(m[flag].astype(bool).to_numpy(),a[flag]):
            stop("C0.8 loader smoke", {"split":split,"flag_mismatch":flag})
        flags[flag]=int(m[flag].sum())
    smoke[split]={"shape":list(a["X"].shape),"finite":True,"groups":int(m.group.nunique()),
                  "labels":{str(k):int(v) for k,v in m.label.value_counts().sort_index().items()},
                  "flag_counts":flags}
idx=pd.read_csv(P["spec"]/"index_train.csv",keep_default_na=False)
eligible=~(idx.silent.astype(bool)|idx.near_silent.astype(bool)|idx.crop_all_zero.astype(bool))
norm=np.load(P["spec"]/"norm_stats_global.npz",allow_pickle=False)
if int(eligible.sum())!=7519 or int(norm["clips_used"])!=7519 or int(norm["clips_excluded"])!=88:
    stop("C0.8 train stats eligibility", {"eligible":int(eligible.sum()),"clips_used":int(norm['clips_used']),"clips_excluded":int(norm['clips_excluded'])})
smoke["train_stats"]={"eligible":int(eligible.sum()),"excluded":int(norm["clips_excluded"]),
                      "frames":int(norm["frame_count"])}
record("C0.8 loader smoke","PASS",smoke)
STATE.update({"manifest_hashes":hashes,"frontend_params_hash":param_hash,
              "path_counts":path_counts,"loader_smoke":smoke,
              "frontend_core_frozen_tag_present":TAG_PRESENT,
              "notebook_30_blocked_until_tag":not TAG_PRESENT})
write_report()

In [ ]:
from google.colab import drive
drive.flush_and_unmount()